### Training MMIDAS - a coupled mixture VAE model
This notebook guides you through the process of training a mixture variational autoencoder.

In [351]:
import os

if os.getcwd().split(os.sep)[-1] != 'mmidas':
    os.chdir('..')

assert os.getcwd().split(os.sep)[-1] == 'mmidas', "Please run this script from the mmidas directory."

In [352]:
%load_ext autoreload
%autoreload 2
from mmidas.cpl_mixvae import cpl_mixVAE
from mmidas.utils.tools import get_paths
from mmidas.utils.dataloader import load_data, get_loaders, show_summary
from mmidas._utils import set_seeds

import warnings
warnings.filterwarnings("ignore")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Specify the training parameters.

In [353]:
spec = {
    "n_run": 1,
    "augmentation": False,
    "n_categories": 120,
    "input_dim": 5032,
    "state_dim": 2,
    "n_arm": 6,
    "latent_dim": 10,
    "batch_size": 5000,
    "n_epoch": 4,
    "n_epoch_p": 0,
    "min_con": 0.9,
    "max_prun_it": 2,
    "batch_size:": 5000,
    "lr": 1e-3,
    "seed": 546,
    "device": "mps"
}

set_seeds(spec['seed'])

Load the prepared data (as described in ```1_data_prep.ipynb```) and create training and validation sets.

In [354]:
data_file = "Mouse_ALM-VISp_cpm.h5ad"
dataset = load_data(".." + "/" + "data" + "/" + data_file)
xs_T = dataset['log1p']
train_loader, test_loader, _, = get_loaders(xs_T, batch_size=spec['batch_size'])

print(show_summary(dataset))

Summary
n_cell_types: 115
n_cells: 22365
n_genes: 5032


Create a designated folder to store training files

In [355]:
def show_spec(spec):
    n_run        = spec['n_run']
    n_categories = spec['n_categories']
    state_dim    = spec['state_dim']
    augmentation = spec['augmentation']
    lr           = spec['lr']
    n_arm        = spec['n_arm']
    batch_size   = spec['batch_size']
    n_epoch      = spec['n_epoch']
    n_epoch_p    = spec['n_epoch_p']
    return "_".join(["run"     + "_" + str(n_run),
                     "K"       + "_" + str(n_categories),
                     "Sdim"    + "_" + str(state_dim),
                     "aug"     + "_" + str(augmentation),
                     "lr"      + "_" + str(lr),
                     "n_arm"   + "_" + str(n_arm),
                     "nbatch"  + "_" + str(batch_size),
                     "nepoch"  + "_" + str(n_epoch),
                     "nepochP" + "_" + str(n_epoch_p)])

saving_folder = "results" + "/" + show_spec(spec)
os.makedirs(saving_folder, exist_ok=True)
os.makedirs(saving_folder + "/" + "model", exist_ok=True)

Construct a cpl-mixVAE object and launch its training on the prepared data.

In [356]:
import torch as th

set_seeds(spec['seed'])

cplMixVAE = cpl_mixVAE(saving_folder=saving_folder, device=spec['device'])
cplMixVAE.init_model(
    n_categories=spec['n_categories'],
    state_dim=spec['state_dim'],
    input_dim=spec['input_dim'],
    lowD_dim=spec['latent_dim'],
    lr=spec['lr'],
    n_arm=spec['n_arm']
)
cplMixVAE.model = cplMixVAE.model.to(spec['device'])

model_file = cplMixVAE.train(
    train_loader=train_loader,
    test_loader=test_loader,
    n_epoch=spec['n_epoch'],
    n_epoch_p=spec['n_epoch_p'],
    min_con=spec['min_con'],
    max_prun_it=spec['max_prun_it']
)

using device: mps
1 epoch = 4 batches
step: 1 | loss: 2559275776.00 | loss_joint: 2557946368.00 | loss_rec_0: 44309.83 | loss_rec_1: 44319.17 | loss_rec_2: 44306.75 | loss_rec_3: 44309.30 | loss_rec_4: 44302.24 | loss_rec_5: 44319.78 | throughput: 644.31 | dt: 7760.23
step: 2 | loss: 1970953984.00 | loss_joint: 1969624832.00 | loss_rec_0: 44299.38 | loss_rec_1: 44314.55 | loss_rec_2: 44306.56 | loss_rec_3: 44301.36 | loss_rec_4: 44289.37 | loss_rec_5: 44312.38 | throughput: 15519.32 | dt: 322.18
step: 3 | loss: 1761418880.00 | loss_joint: 1760091264.00 | loss_rec_0: 44244.79 | loss_rec_1: 44264.98 | loss_rec_2: 44262.84 | loss_rec_3: 44248.75 | loss_rec_4: 44230.77 | loss_rec_5: 44260.49 | throughput: 19413.42 | dt: 257.55
step: 4 | loss: 1503009408.00 | loss_joint: 1501682432.00 | loss_rec_0: 44220.64 | loss_rec_1: 44245.52 | loss_rec_2: 44250.76 | loss_rec_3: 44226.20 | loss_rec_4: 44201.57 | loss_rec_5: 44239.93 | throughput: 18249.24 | dt: 273.98
step: 5 | loss: 1318513664.00 | los

In [357]:
from mmidas.train import mmidas_train
from mmidas.model import _make_mmidas, make_mmidas, module_n_params, module_params, make_mspec, mspec_lookup

os.environ['WANDB_MODE'] = 'disabled'

# TODO: enhance: more descriptive variable names
es = { # Experiment specification
    'model': 'MMIDAS',
    'solver': 'adam',
    'dataset': 'Mouse_ALM-VISp_cpm',
    'batch_size': 5000,
    'backend': 'torch',
    'is_augment': False,
    'input_dim': 5032,
    'fc_dim': 100,
    'lowD_dim': 10,
    'state_dim': 2,
    'n_categories': 120,
    'n_arms': 6,
    'n_pr': 0,
    'n_epochs': 4,
    'temp': 1.0,
    'eps': 1e-8,
    'is_ref_prior': False,
    'x_drop': 0.5,
    's_drop': 0.2,
    'lam': 1,
    'lam_pc': 1,
    'tau': 0.005,
    'beta': 1.0,
    'is_hard': False,
    'is_variational': True,
    'momentum': 0.01,
    'loss': 'MSE',
    'lr': 1e-3,
    'c_prior': 0,
    'c_onehot': 0,
    'min_con': 0.5,
    'max_prun_it': 0,
    'print_every': 1,
    'device': 'mps',
    'seed': 546
}

device = 'mps'
mmidas_spec = make_mspec(n_arms=es['n_arms'], device=device)

set_seeds(es['seed'])

if es['model'] == 'mixVAE_model':
    model = _make_mmidas({**make_mspec(), 'device': es['device']}).to(es['device'])
elif es['model'] == 'MMIDAS':
    model = make_mmidas(mmidas_spec).to(device)

if es['solver'] == 'adam':
    solver = th.optim.Adam(model.parameters(), lr=es['lr'])
elif es['solver'] == 'adamw':
    solver = th.optim.AdamW(model.parameters(), lr=es['lr'])
else:
    assert False, "Unknown solver: {}".format(es['solver'])

es['n_params'] = module_n_params(model)

mmidas_train(model, solver, train_loader, test_loader, es)

# th.save(module_params(model), es['model'] + "_" + "epochs" + str(es['n_epochs']) + "_" + es['solver'] + '_' + 'lr' + str(es['lr']) + ".pt")

using device: mps
1 epoch = 4 batches
step: 1 | loss: 2559275776.00 | loss_joint: 2557946368.00 | loss_rec_0: 44309.83 | loss_rec_1: 44319.17 | loss_rec_2: 44306.75 | loss_rec_3: 44309.30 | loss_rec_4: 44302.24 | loss_rec_5: 44319.78 | throughput: 16366.07 | dt: 305.51
step: 2 | loss: 1970953984.00 | loss_joint: 1969624832.00 | loss_rec_0: 44299.38 | loss_rec_1: 44314.55 | loss_rec_2: 44306.56 | loss_rec_3: 44301.36 | loss_rec_4: 44289.37 | loss_rec_5: 44312.38 | throughput: 12687.20 | dt: 394.10
step: 3 | loss: 1761418880.00 | loss_joint: 1760091264.00 | loss_rec_0: 44244.79 | loss_rec_1: 44264.98 | loss_rec_2: 44262.84 | loss_rec_3: 44248.75 | loss_rec_4: 44230.77 | loss_rec_5: 44260.49 | throughput: 17560.25 | dt: 284.73
step: 4 | loss: 1503009408.00 | loss_joint: 1501682432.00 | loss_rec_0: 44220.64 | loss_rec_1: 44245.52 | loss_rec_2: 44250.76 | loss_rec_3: 44226.20 | loss_rec_4: 44201.57 | loss_rec_5: 44239.93 | throughput: 19947.57 | dt: 250.66
step: 5 | loss: 1318513664.00 | lo

Working directly with command line, you have the option to train the model using a Python file, such as ```tutorial/train_unimodal.py``` as follows.

```
python train_unimodal.py --n_epoch 10 --n_epoch_p 5 --max_prun_it 2
```
or
```
python train_unimodal.py --n_epoch 10 --n_epoch_p 5 --max_prun_it 2 --device 'cuda'
```